In [1]:
import pandas as pd

# Getting the Scoring Sequence for each divison

In [5]:
nba_divisions = {
    "Atlantic":  ("BOS", "BKN", "NYK", "PHI", "TOR"),
    "Central":   ("CHI", "CLE", "DET", "IND", "MIL"),
    "Southeast": ("ATL", "CHA", "MIA", "ORL", "WAS"),
    "Northwest": ("DEN", "MIN", "OKC", "POR", "UTA"),
    "Pacific":   ("GSW", "LAC", "LAL", "PHX", "SAC"),
    "Southwest": ("DAL", "HOU", "MEM", "NOP", "SAS")
}

rows = []

for season in range(2004, 2025):
    for division, teams in nba_divisions.items():

        teams = list(teams)

        if season <= 2011 and "BKN" in teams:
            teams[teams.index("BKN")] = "NJN"

        if season <= 2007 and "OKC" in teams:
            teams[teams.index("OKC")] = "SEA"

        if "NOP" in teams:
            if season in [2005, 2006]:
                teams[teams.index("NOP")] = "NOK"
            elif season <= 2012:
                teams[teams.index("NOP")] = "NOH"

        rows.append({
            "season": season,
            "league": "NBA",
            "division": division,
            "division_teams": tuple(teams)
        })

division_df = pd.DataFrame(rows)

games_df = pd.read_csv("../Data/game_data_us_leagues.csv")

# Make one row per season/team/division
team_division_df = (
    division_df
    .explode("division_teams")
    .rename(columns={"division_teams": "team"})
)

# this is only the NBA games
nba_games = games_df[
    (games_df["league"] == "NBA") &
    (games_df["season"].between(2004, 2024))
].copy()

# adding team 1 and team 2 division
nba_games = nba_games.merge(
    team_division_df[["season", "team", "division"]],
    left_on=["season", "team1"],
    right_on=["season", "team"],
    how="left"
)

nba_games = (
    nba_games
    .rename(columns={"division": "division1"})
    .drop(columns="team")
)

# adds team 2 div
nba_games = nba_games.merge(
    team_division_df[["season", "team", "division"]],
    left_on=["season", "team2"],
    right_on=["season", "team"],
    how="left"
)

nba_games = (
    nba_games
    .rename(columns={"division": "division2"})
    .drop(columns="team")
)

# FILTERING BY DIVISION
div_games = nba_games[
    nba_games["division1"] == nba_games["division2"]
].copy()

div_games["division"] = div_games["division1"]

div_games = div_games.drop(columns=["division1", "division2"])

# go game by game and figure out the winner
def get_winner(row):
    if row["score1"] > row["score2"]:
        return row["team1"]
    elif row["score2"] > row["score1"]:
        return row["team2"]
    else:
        return None


div_games["winner"] = div_games.apply(
    get_winner,
    axis=1
)

# pairs need to have same ordering so when we do pd.group we dont double count
teams = div_games[["team1", "team2"]]

div_games["team1"] = teams.min(axis=1)
div_games["team2"] = teams.max(axis=1)

# mark whoever winner in a div game
div_games["win1"] = (
    div_games["winner"] == div_games["team1"]
).astype(int)

div_games["win2"] = (
    div_games["winner"] == div_games["team2"]
).astype(int)

# compute the series games
series_df = (
    div_games
    .groupby(
        ["season", "division", "team1", "team2"],
        as_index=False
    )
    .agg(
        wins1=("win1", "sum"),
        wins2=("win2", "sum")
    )
)

# determine the full series winner
def get_series_result(row):
    if row["wins1"] > row["wins2"]:
        return 1
    elif row["wins2"] > row["wins1"]:
        return -1
    else:
        return 0


series_df["result"] = series_df.apply(
    get_series_result,
    axis=1
)

series_df.insert(
    1,
    "league",
    "NBA"
)

series_df = series_df[
    [
        "season",
        "league",
        "division",
        "team1",
        "team2",
        "result",
        "wins1",
        "wins2"
    ]
]

# GET THE SERIES WINS FOR EACH TEAM

team1 = series_df[
    ["season", "league", "division", "team1", "result"]
].copy()

team1 = team1.rename(columns={"team1": "team"})

team1["wins"] = (
    team1["result"] == 1
).astype(int)

team2 = series_df[
    ["season", "league", "division", "team2", "result"]
].copy()

team2 = team2.rename(columns={"team2": "team"})

team2["wins"] = (
    team2["result"] == -1
).astype(int)

# create scoring sequences
score_sequences = pd.concat(
    [team1, team2],
    ignore_index=True
)

score_sequences = (
    score_sequences
    .groupby(
        ["season", "league", "division", "team"],
        as_index=False
    )["wins"]
    .sum()
)

score_sequences = score_sequences.sort_values(
    ["season", "division", "wins"]
)

In [8]:
score_sequences.to_csv('scoring_sequence.csv', index=False)

# Getting upset frequencies per division

In [9]:
# lookup each team's score for a given season and division
score_lookup = score_sequences.set_index(
    ["season", "division", "team"]
)["wins"].to_dict()


# determine whether each series is an upset
def is_upset(row):
    if row["result"] == 0:
        return 0

    if row["result"] == 1:
        winner = row["team1"]
        loser = row["team2"]
    else:
        winner = row["team2"]
        loser = row["team1"]

    winner_score = score_lookup[
        (row["season"], row["division"], winner)
    ]

    loser_score = score_lookup[
        (row["season"], row["division"], loser)
    ]

    return int(winner_score < loser_score)


series_df["upset"] = series_df.apply(
    is_upset,
    axis=1
)


# count upsets for each season and division
upset_counts = (
    series_df
    .groupby(
        ["season", "league", "division"],
        as_index=False
    )["upset"]
    .sum()
)

In [11]:
upset_counts.to_csv('upset_counts.csv', index=False)